# AREPA — CRISP-DM: Salud — nivel de glucosa

Pipeline: datos → limpieza → EDA → regresión lineal → evaluación (MSE/R²) → exportación `.joblib`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from arepa.constants import FIGURES_DIR
from arepa.data_pipeline import (
    audit_data,
    check_multicollinearity,
    clean_data,
    detect_outliers,
    load_data,
)
from arepa.modeling import (
    fit_ols_inference,
    get_coefficients,
    split_data,
    train_scenario,
    export_model,
)
from arepa.visualization import plot_correlation_heatmap, plot_feature_relationship

SCENARIO = 'glucose'
TARGET = 'glucose_level'
FEATURES = ['age', 'bmi', 'physical_activity_hours']


## 1. Comprensión de datos

Cargamos el CSV, renombramos columnas a inglés (`snake_case`) y auditamos nulos, tipos y duplicados.

In [ ]:
raw = load_data(SCENARIO)
audit_data(raw)

## 2. Preparación de datos

Limpieza, encoding cíclico de `hour` (energía) y flags `is_outlier_*` sin eliminar filas (criterio documentado en el informe).

In [ ]:
df = clean_data(raw, SCENARIO)
df = detect_outliers(df)
df.head()

In [ ]:
outlier_count = int(df['is_outlier'].sum()) if 'is_outlier' in df.columns else 0
print(f'Outlier rows (kept): {outlier_count}')

In [ ]:
check_multicollinearity(df, FEATURES)

## 3. Visualización

Relación de cada predictor con la variable objetivo + heatmap de correlación.

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_feats = ['age', 'bmi', 'physical_activity_hours']
for feat in plot_feats:
    if feat in df.columns:
        plot_feature_relationship(df, feat, TARGET, save_path=FIGURES_DIR / f'{SCENARIO}_{feat}_vs_{TARGET}.png')
        plt.show()
plot_correlation_heatmap(df, plot_feats + [TARGET], save_path=FIGURES_DIR / f'{SCENARIO}_correlation.png')
plt.show()

## 4. Modelado

`StandardScaler` + `LinearRegression`, split 80/20 (`random_state=42`). Comparamos métricas con y sin filas outlier.

In [ ]:
artifact = train_scenario(df, SCENARIO, exclude_outliers=False)
artifact_no_out = train_scenario(df, SCENARIO, exclude_outliers=True)
metrics = artifact['metrics']
print('MSE:', metrics['mse'], 'R²:', metrics['r2'], 'RMSE:', metrics.get('rmse'))
print('Excluding outliers:', artifact_no_out['metrics'])

In [ ]:
get_coefficients(artifact['pipeline'], FEATURES)

In [ ]:
x_train, x_test, y_train, y_test = split_data(df, TARGET, FEATURES)
ols = fit_ols_inference(x_train, y_train)
print(ols.summary())

## 5. Despliegue (artefacto)

Exportamos pipeline + metadatos para el API FastAPI (Sprint 2).

In [ ]:
export_model(
    {
        'pipeline': artifact['pipeline'],
        'feature_names': FEATURES,
        'target': TARGET,
        'metrics': metrics,
        'scenario': SCENARIO,
    },
    scenario=SCENARIO,
)
print('Exported to models/')